In [1]:
import pandas as pd

In [31]:
import janitor

janitor

import geopandas as gpd

In [32]:
# Subzones/\nbhs{} (finer)
gdf_sz = gpd.read_file(
    "../build/raw-data/master-plan-2019-subzone-boundary-no-sea/master-plan-2019-subzone-boundary-no-sea-kml.kml"
)
gdf_sz

/home/lsys/nbhmv/venv_nbhmv/lib/python3.10/site-packages/pyogrio/__init__.py:7: DeprecationWarning: The 'shapely.geos' module is deprecated, and will be removed in a future version. All attributes of 'shapely.geos' are available directly from the top-level 'shapely' namespace (since shapely 2.0.0).
  import shapely.geos  # noqa: F401


,Name,Description,geometry
0,kml_1,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.88025 1.28386 0, 103.88024 1.2..."
1,kml_2,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.83764 1.2956 0, 103.83766 1.29..."
2,kml_3,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.8341 1.29248 0, 103.83411 1.29..."
3,kml_4,<center><table><tr><th colspan='2' align='cent...,"MULTIPOLYGON Z (((103.71253 1.29163 0, 103.712..."
4,kml_5,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.84718 1.297 0, 103.84728 1.296..."
...,...,...,...
327,kml_328,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.84263 1.36323 0, 103.84244 1.3..."
328,kml_329,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.83537 1.37093 0, 103.83576 1.3..."
329,kml_330,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.84814 1.36879 0, 103.84834 1.3..."
330,kml_331,<center><table><tr><th colspan='2' align='cent...,"POLYGON Z ((103.84652 1.34521 0, 103.84649 1.3..."


In [64]:
# Census areas, with residential records
df_census = (
    pd.read_csv(
        "../build/raw-data/resident-population-by-planning-area-subzone-age-group-and-sex-2015/resident-population-by-planning-area-age-group-and-sex.csv"
    )
    .rename_column("level_3", "pa")
    .groupby("pa")["value"]
    .sum()
    .reset_index()
    .query("pa!='Total'")
    .reset_index(drop=True)
    .assign(pa=lambda df_: df_["pa"].str.replace("- Total", ""))
)
df_census

/home/lsys/nbhmv/venv_nbhmv/lib/python3.10/site-packages/pandas_flavor/register.py:164: FutureWarning: This function will be deprecated in a 1.x release. Please use `pd.DataFrame.rename` instead.
  return method(self._obj, *args, **kwargs)


,pa,value
0,Ang Mo Kio,699080
1,Bedok,1159000
2,Bishan,362820
3,Boon Lay,30
4,Bukit Batok,557080
5,Bukit Merah,623370
6,Bukit Panjang,556160
7,Bukit Timah,297950
8,Central Water Catchment,10
9,Changi,10130


In [33]:
df = pd.read_parquet("input/cd-panel-labels.parquet").dropna(subset=["inflow"])
df.head(3)

,date,home,inflow,basecount,dest_basecount,area,pa_home,htype_home,resident_count_home,private_home,...,private_price_home,private_num_transactions_dest,private_psm_dest,private_price_dest,wpsm_private_home,wpsm_public_home,wpsm_home,wpsm_private_dest,wpsm_public_dest,wpsm_dest
0,2020-01-01,admiralty,14.583333,48,24,admiralty,Sembawang,HDB 1- and 2-Room FlatsHDB 3-Room FlatsHDB 4-R...,14110,0.0,...,NaN,NaN,NaN,NaN,NaN,3930.897154,3930.897154,NaN,3930.897154,3930.897154
1,2020-01-01,admiralty,2.083333,48,83,anson,Sembawang,HDB 1- and 2-Room FlatsHDB 3-Room FlatsHDB 4-R...,14110,0.0,...,NaN,31.0,20121.516129,1.192871e+06,NaN,3930.897154,3930.897154,NaN,NaN,NaN
2,2020-01-01,admiralty,2.083333,48,1125,balestier,Sembawang,HDB 1- and 2-Room FlatsHDB 3-Room FlatsHDB 4-R...,14110,0.0,...,NaN,975.0,14746.951795,1.452991e+06,NaN,3930.897154,3930.897154,4673.847113,3244.201532,7918.048645


In [35]:
# Census areas, with GPS records
df["pa_home"].nunique()

52

In [38]:
# Census areas, with GPS records
df["pa_dest"].nunique()

52

In [39]:
df["home"].nunique()

297

In [41]:
# Subzones/\nbhs{}, with GPS records
df["area"].nunique()

300

In [51]:
(
    df.drop_duplicates(["home"], ignore_index=True)
    .query("resident_count_home>0")["home"]
    .nunique()
)

219

In [67]:
c = 0
for pa in list(df["pa_home"].unique()):
    if pa in df_census["pa"].tolist():
        c += 1
    else:
        print(pa)
print(c)

Western Islands
Marina East
Marina South
Simpang
Straits View
47


In [50]:
df_census_sz = (
    pd.read_csv(
        "../build/raw-data/resident-population-by-planning-area-subzone-age-group-and-sex-2015/resident-population-by-subzone-age-group-and-sex.csv"
    )
    .rename_column("level_4", "sz")
    .groupby("sz")["value"]
    .sum()
    .reset_index()
    .query("value>0")
    .reset_index(drop=True)
)
df_census_sz

/home/lsys/nbhmv/venv_nbhmv/lib/python3.10/site-packages/pandas_flavor/register.py:164: FutureWarning: This function will be deprecated in a 1.x release. Please use `pd.DataFrame.rename` instead.
  return method(self._obj, *args, **kwargs)


,sz,value
0,Admiralty,57640
1,Alexandra Hill,62630
2,Alexandra North,3990
3,Aljunied,166840
4,Anak Bukit,87970
...,...,...
276,Yishun South,155370
277,Yishun West,231560
278,Yuhua East,109890
279,Yuhua West,83530


In [43]:
# Subzones/\nbhs{}, with residential records
df_census_sz["sz"].nunique()

281